In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    recall_score,
    precision_score,
    roc_auc_score,
    average_precision_score
)


In [3]:

RANDOM_STATE = 12345

# 1) Features y target usando tus variables del notebook
X = df_sample[X_cols].copy()
y = df_sample[target_col].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

# 2) Separar numéricas y categóricas
num_features = X_train.select_dtypes(include=["number"]).columns.tolist()
cat_features = [c for c in X_train.columns if c not in num_features]

print("Num features:", len(num_features))
print("Cat features:", len(cat_features))
print("Train size:", X_train.shape, "Test size:", X_test.shape)

# 3) Preprocesamiento
num_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

cat_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_features),
        ("cat", cat_pipe, cat_features)
    ],
    remainder="drop"
)

# 4) Modelo baseline
model = LogisticRegression(
    max_iter=2000,
    solver="saga",
    class_weight="balanced",
    random_state=RANDOM_STATE
)

clf = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", model)
    ]
)

# 5) Fit
clf.fit(X_train, y_train)

# 6) Predicción con probas para poder mover umbral
proba_test = clf.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= 0.5).astype(int)

# 7) Métricas
cm = confusion_matrix(y_test, pred_test, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

recall_1 = recall_score(y_test, pred_test, pos_label=1)
precision_1 = precision_score(y_test, pred_test, pos_label=1, zero_division=0)

roc_auc = roc_auc_score(y_test, proba_test)
pr_auc = average_precision_score(y_test, proba_test)

approved = (pred_test == 0).sum()
total = len(pred_test)
approval_rate = approved / total

# malos aprobados = reales 1 predichos 0
bad_approved = fn
bad_approval_rate_among_approved = bad_approved / max(approved, 1)

print("\nConfusion matrix labels [0,1]:")
print(cm)

print("\nReporte:")
print(classification_report(y_test, pred_test, digits=4))

print("\nMétricas clave negocio:")
print("Recall clase 1:", round(recall_1, 4))
print("Precision clase 1:", round(precision_1, 4))
print("ROC AUC:", round(roc_auc, 4))
print("PR AUC:", round(pr_auc, 4))
print("Approval rate (pred 0):", round(approval_rate, 4))
print("Bad approvals (FN):", int(bad_approved))
print("Bad approval rate among approved:", round(bad_approval_rate_among_approved, 4))


NameError: name 'df_sample' is not defined

In [ ]:
def metrics_at_threshold(y_true, proba, thr):
    pred = (proba >= thr).astype(int)
    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    recall_1 = recall_score(y_true, pred, pos_label=1)
    precision_1 = precision_score(y_true, pred, pos_label=1, zero_division=0)

    approved = (pred == 0).sum()
    total = len(pred)
    approval_rate = approved / total

    bad_approved = fn
    bad_approval_rate_among_approved = bad_approved / max(approved, 1)

    return {
        "threshold": thr,
        "recall_1": recall_1,
        "precision_1": precision_1,
        "approval_rate": approval_rate,
        "bad_approved": bad_approved,
        "bad_approval_rate_among_approved": bad_approval_rate_among_approved
    }

thresholds = np.round(np.linspace(0.1, 0.9, 17), 2)
rows = [metrics_at_threshold(y_test, proba_test, t) for t in thresholds]
thr_table = pd.DataFrame(rows).sort_values("recall_1", ascending=False)

thr_table.head(10)


In [ ]:
# ejemplo: mantener al menos 40% de aprobaciones y maximizar recall de 1
thr_table_filtered = thr_table[thr_table["approval_rate"] >= 0.40].sort_values("recall_1", ascending=False)
thr_table_filtered.head(10)
